# تدريب التعرف على الوجوه باستخدام EfficientNetV2-M

هذا الدفتر مخصص لتدريب موديل التعرف على الوجوه باستخدام معمارية **EfficientNetV2-M** القوية. تأكد من تفعيل الـ GPU من خلال `Runtime` -> `Change runtime type` -> `Hardware accelerator` -> `T4 GPU`.



### 💡 لماذا EfficientNetV2-M؟
تعتبر هذه المعمارية من أحدث وأقوى نماذج الرؤية الحاسوبية، حيث توفر:
* **دقة عالية:** بفضل تقنيات البحث عن المعمارية العصبية (NAS).
* **سرعة في التدريب:** تم تحسينها لتعمل بكفاءة على وحدات معالجة الرسوميات (GPUs).
* **كفاءة المعلمات:** تعطي نتائج مذهلة بعدد أقل من المعلمات مقارنة بالنماذج التقليدية الضخمة.

---

# Face Recognition Training with EfficientNetV2-M

This notebook is dedicated to training a face recognition model using the powerful **EfficientNetV2-M** architecture. Ensure the GPU is enabled via `Runtime` -> `Change runtime type` -> `Hardware accelerator` -> `T4 GPU`.

### 💡 Why EfficientNetV2-M?
This architecture is one of the latest and most robust computer vision models, providing:
* **High Accuracy:** Thanks to Neural Architecture Search (NAS) techniques.
* **Training Speed:** Optimized for efficient execution on GPUs.
* **Parameter Efficiency:** Delivers stunning results with fewer parameters compared to traditional large-scale models.

1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

2. Importing Libraries and Initializing Paths

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import matplotlib.pyplot as plt
import zipfile

# --- المسارات ---
# قم بتعديل هذا المسار ليطابق مجلد مشروعك في Drive
PROJECT_PATH = '/content/drive/MyDrive/My Face Recogniation Project'
DATASET_ZIP = os.path.join(PROJECT_PATH, 'dataset.zip')

print("TensorFlow version:", tf.__version__)
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  print('GPU device not found')
else:
  print('Found GPU at: {}'.format(device_name))

3. Data Preparation (Unzipping)
It is always preferred to upload data as a single ZIP file for faster reading speeds in Colab.

In [ ]:
if os.path.exists(DATASET_ZIP):
    print("Unzipping dataset...")
    with zipfile.ZipFile(DATASET_ZIP, 'r') as zip_ref:
        zip_ref.extractall('/content/dataset_local')
    DATA_DIR = '/content/dataset_local'
else:
    print("ZIP file not found, using direct path (might be slower)...")
    DATA_DIR = os.path.join(PROJECT_PATH, 'dataset')

print(f"Data directory set to: {DATA_DIR}")

4. Data Loading and Augmentation Settings

In [ ]:
IMG_SIZE = (380, 380) 
BATCH_SIZE = 16 

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print(f"Classes: {class_names}")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

data_augmentation = tf.keras.Sequential([
  layers.RandomFlip('horizontal'),
  layers.RandomRotation(0.2),
  layers.RandomZoom(0.2),
])

5. Building the Model (EfficientNetV2-M)

In [ ]:
base_model = tf.keras.applications.EfficientNetV2M(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = True 

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = base_model(x, training=True) 
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
embeddings = layers.Dense(256, activation=None, name='embedding_layer')(x) 
outputs = layers.Dense(len(class_names), activation='softmax')(embeddings)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

model.summary()

6. Start Training

In [ ]:
checkpoint_path = os.path.join(PROJECT_PATH, "my_face_model_v2m.keras")

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    checkpoint_path, 
    monitor='val_accuracy', 
    save_best_only=True, 
    mode='max', 
    verbose=1
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', 
    patience=7, 
    restore_best_weights=True
)

EPOCHS = 30
history = model.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stopping]
)

7. Results and Visualization

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Accuracy History')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.history['loss'], label='Training Loss')
plt.plot(epochs_range, history.history['val_loss'], label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Loss History')
plt.show()